## Setup

In [ ]:
import sys
import os
import importlib

from dotenv import load_dotenv

sys.path.append("../src")

import utils
import metrics

# Reload modules to apply any changes
importlib.reload(utils)
importlib.reload(metrics)

# NOTE: define variables in .env to avoid changing the notebook
load_dotenv("../.env")

In [ ]:
FILENAME = os.getenv("FILENAME", "parkinson")
BACKEND = os.getenv("BACKEND", "openai")
CUMULATIVE = os.getenv("CUMULATIVE", "true").lower() == "true"

if not CUMULATIVE:
    DST = f"../results/{BACKEND}-noncum/{FILENAME}"
else:
    DST = f"../results/{BACKEND}/{FILENAME}"

print(f"{FILENAME=}")
print(f"{BACKEND=}")
print(f"{CUMULATIVE=}")
print(f"{DST=}")

df = utils.load(f"../data/embeddings/{BACKEND}/{FILENAME}.csv")

print(f"{df.shape=}")
df.head(3)

## Metrics

In [ ]:
# NOTE: this cell enables ZCA whitening
# df, zca_params = utils.zca_whitened_embeddings(
#     df, emb_col="embedding", new_col="embedding_zca"
# )
# df = df.drop(columns=["embedding"]).rename(columns={"embedding_zca": "embedding"})
# df.head(3)

In [ ]:
# Compute geometric metrics: distance to next, entropy, and distance to centroid
df = df.groupby(["id", "concept"], group_keys=False)
df = df.apply(metrics.geometric, include_groups=True, cumulative=CUMULATIVE)
df = df.reset_index(drop=True)

In [ ]:
# Compute kinematic metrics: velocity and acceleration
grouped = df.groupby(["id", "concept"], group_keys=True)
df = grouped.apply(metrics.kinematic, include_groups=True, cumulative=CUMULATIVE)
df = df.reset_index(drop=True)

In [ ]:
# Keep only original columns + computed metrics (not embeddings)
to_drop = ("prop_embedding", "properties_cum", "embedding", "vel_vector", "acc_vector")
cols = [col for col in df.columns if col not in to_drop]
df = df[cols]

mlist = ["entropy", "d_next", "d_centroid", "vel", "acc"]
print(f"NaN values:\n\n{df[mlist].isna().mean()}\n")

# Inspect sample of results
df.head(3)

In [ ]:
# Save results
os.makedirs(DST, exist_ok=True)
df[mlist].isna().mean().to_csv(f"{DST}/metrics-nan.csv")
utils.save(df, f"{DST}/metrics.csv")
print(f"{df.shape=}")